In [ ]:
!pip install yfinance

In [19]:
#Set up project & install yfinance, statsmodels, prophet
import yfinance as yf
import pandas as pd
import statsmodels as stats
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [25]:
#Fetch 5-year OHLCV data for AAPL, MSFT, and SPY
data = yf.download(["AAPL", "MSFT", "SPY"], period="5y", auto_adjust=False)

[*********************100%***********************]  3 of 3 completed


In [45]:
data.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1254 entries, 2021-06-25 to 2026-06-24
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   (Adj Close, AAPL)  1254 non-null   float64
 1   (Adj Close, MSFT)  1254 non-null   float64
 2   (Adj Close, SPY)   1254 non-null   float64
 3   (Close, AAPL)      1254 non-null   float64
 4   (Close, MSFT)      1254 non-null   float64
 5   (Close, SPY)       1254 non-null   float64
 6   (High, AAPL)       1254 non-null   float64
 7   (High, MSFT)       1254 non-null   float64
 8   (High, SPY)        1254 non-null   float64
 9   (Low, AAPL)        1254 non-null   float64
 10  (Low, MSFT)        1254 non-null   float64
 11  (Low, SPY)         1254 non-null   float64
 12  (Open, AAPL)       1254 non-null   float64
 13  (Open, MSFT)       1254 non-null   float64
 14  (Open, SPY)        1254 non-null   float64
 15  (Volume, AAPL)     1254 non-null   int64  
 16  (Volum

In [26]:
data.head()

Price        Adj Close                               Close              \
Ticker            AAPL        MSFT         SPY        AAPL        MSFT   
Date                                                                     
2021-06-25  129.749207  254.338791  398.722809  133.110001  265.019989   
2021-06-28  131.377060  257.889771  399.526611  134.779999  268.720001   
2021-06-29  132.887924  260.461700  399.741608  136.330002  271.399994   
2021-06-30  133.502029  259.981873  400.078064  136.960007  270.899994   
2021-07-01  133.804199  260.653534  402.293091  137.270004  271.600006   

Price                         High                                 Low  \
Ticker             SPY        AAPL        MSFT         SPY        AAPL   
Date                                                                     
2021-06-25  426.609985  133.889999  267.250000  427.089996  132.809998   
2021-06-28  427.470001  135.250000  268.899994  427.649994  133.350006   
2021-06-29  427.700012  136.490005  271.649994  428.559998  134.350006   
2021-06-30  428.059998  137.410004  271.359985  428.779999  135.869995   
2021-07-01  430.429993  137.330002  271.839996  430.600006  135.759995   

Price                                     Open                          \
Ticker            MSFT         SPY        AAPL        MSFT         SPY   
Date                                                                     
2021-06-25  264.760010  425.549988  133.460007  266.230011  425.899994   
2021-06-28  265.910004  425.890015  133.410004  266.190002  427.170013   
2021-06-29  267.980011  427.130005  134.800003  268.869995  427.880005   
2021-06-30  269.600006  427.179993  136.169998  270.690002  427.209991   
2021-07-01  269.600006  428.799988  136.600006  269.609985  428.869995   

Price         Volume                      
Ticker          AAPL      MSFT       SPY  
Date                                      
2021-06-25  70783700  25611100  58129500  
2021-06-28  62111300  19590000  53159600  
2021-06-29  64556100  19937800  35970500  
2021-06-30  63261400  21656500  64827900  
2021-07-01  52485800  16725300  53441000

In [27]:
data.isna().sum().sum()

np.int64(0)

In [28]:
#Handle non-trading days & forward-fill missing data
price_cols = price_cols = ['Open','High','Low','Close']
data[price_cols].ffill()

Price             Open                                High              \
Ticker            AAPL        MSFT         SPY        AAPL        MSFT   
Date                                                                     
2021-06-25  133.460007  266.230011  425.899994  133.889999  267.250000   
2021-06-28  133.410004  266.190002  427.170013  135.250000  268.899994   
2021-06-29  134.800003  268.869995  427.880005  136.490005  271.649994   
2021-06-30  136.169998  270.690002  427.209991  137.410004  271.359985   
2021-07-01  136.600006  269.609985  428.869995  137.330002  271.839996   
...                ...         ...         ...         ...         ...   
2026-06-17  300.850006  390.250000  751.289978  302.070007  390.369995   
2026-06-18  298.109985  377.820007  747.760010  300.570007  381.369995   
2026-06-22  297.309998  375.739990  747.700012  302.420013  381.630005   
2026-06-23  297.540009  372.380005  733.809998  301.640015  377.220001   
2026-06-24  295.359985  371.570007  735.169983  299.700012  378.880005   

Price                          Low                               Close  \
Ticker             SPY        AAPL        MSFT         SPY        AAPL   
Date                                                                     
2021-06-25  427.089996  132.809998  264.760010  425.549988  133.110001   
2021-06-28  427.649994  133.350006  265.910004  425.890015  134.779999   
2021-06-29  428.559998  134.350006  267.980011  427.130005  136.330002   
2021-06-30  428.779999  135.869995  269.600006  427.179993  136.960007   
2021-07-01  430.600006  135.759995  269.600006  428.799988  137.270004   
...                ...         ...         ...         ...         ...   
2026-06-17  752.150024  294.359985  377.320007  739.219971  295.950012   
2026-06-18  748.229980  295.619995  373.279999  743.859985  298.010010   
2026-06-22  750.179993  296.760010  367.070007  743.130005  297.010010   
2026-06-23  739.630005  294.179993  370.670013  732.299988  294.299988   
2026-06-24  739.950012  292.940002  364.779999  730.840027  293.079987   

Price                               
Ticker            MSFT         SPY  
Date                                
2021-06-25  265.019989  426.609985  
2021-06-28  268.720001  427.470001  
2021-06-29  271.399994  427.700012  
2021-06-30  270.899994  428.059998  
2021-07-01  271.600006  430.429993  
...                ...         ...  
2026-06-17  378.910004  740.960022  
2026-06-18  379.399994  746.739990  
2026-06-22  367.339996  744.390015  
2026-06-23  373.940002  733.580017  
2026-06-24  365.459991  733.239990  

[1254 rows x 12 columns]

In [34]:
data.columns

MultiIndex([( 'Adj Close', 'AAPL'),
            ( 'Adj Close', 'MSFT'),
            ( 'Adj Close',  'SPY'),
            (     'Close', 'AAPL'),
            (     'Close', 'MSFT'),
            (     'Close',  'SPY'),
            (      'High', 'AAPL'),
            (      'High', 'MSFT'),
            (      'High',  'SPY'),
            (       'Low', 'AAPL'),
            (       'Low', 'MSFT'),
            (       'Low',  'SPY'),
            (      'Open', 'AAPL'),
            (      'Open', 'MSFT'),
            (      'Open',  'SPY'),
            (    'Volume', 'AAPL'),
            (    'Volume', 'MSFT'),
            (    'Volume',  'SPY'),
            ('Log_Return',     '')],
           names=['Price', 'Ticker'])

In [37]:
#Calculate daily logarithmic returns & adjusted close
log_returns = np.log(data["Adj Close"] / data["Adj Close"].shift(1))
log_returns = log_returns.dropna()
log_returns.head()

Ticker,AAPL,MSFT,SPY
Date,,,
2021-06-28,0.012468,0.013865,0.002014
2021-06-29,0.011435,0.009924,0.000538
2021-06-30,0.004611,-0.001844,0.000841
2021-07-01,0.002261,0.002580,0.005521
2021-07-02,0.019407,0.022032,0.007615
